# Notebook 02 — Розвідувальний аналіз даних (EDA)

Мета: зрозуміти структуру транспортної мережі Києва перед побудовою індексу доступності.

Аналізуємо:
- Розподіл зупинок по місту
- Структуру маршрутів (тип, кількість)
- Інтенсивність руху по годинах
- Покриття лікарень та шкіл

In [ ]:
from config_loader import KYIV_CENTER, cfg
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import folium
import warnings
import json
warnings.filterwarnings('ignore')

# Стиль графіків
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11
sns.set_theme(style='whitegrid')

print('Бібліотеки завантажено')
import osmnx as ox



In [ ]:
GTFS_PATH = cfg['paths']['gtfs']
gpkg_data_path = '../data/osm/osm_data.gpkg'


# GTFS дані
gtfs_routes = pd.read_parquet('../data/processed/routes.parquet')
gtfs_trips = pd.read_parquet('../data/processed/trips.parquet')
gtfs_stop_times = pd.read_parquet('../data/processed/stop_times.parquet')

gtfs: dict[str, pd.DataFrame] = {
    'routes': gtfs_routes,
    'trips': gtfs_trips,
    'stop_times': gtfs_stop_times
}

# OSM дані
osm_gstops = gpd.read_file(gpkg_data_path, layer='stops')
osm_ghospitals = gpd.read_file(gpkg_data_path, layer='hospitals')
osm_gschools = gpd.read_file(gpkg_data_path, layer='schools')
osm_gmetro = gpd.read_file(gpkg_data_path, layer='metro')
osm_gtram = gpd.read_file(gpkg_data_path, layer='tram')

osm: dict[str, gpd.GeoDataFrame] = {
    'gstops': osm_gstops,
    'ghospitals': osm_ghospitals,
    'gschools': osm_gschools,
    'gmetro': osm_gmetro,
    'gtram': osm_gtram
}

gtfs_info = json.load(open(f"{GTFS_PATH}gtfs_info.json", 'r'))


print(f'Зупинки OSM (фізичні):          {len(osm_gstops)}')
print(f'Станції метро:                  {len(osm_gmetro)}')
print(f'Маршрути:                       {len(gtfs_routes)}')
print(f'Рейси:                          {len(gtfs_trips)}')
print(f'Лікарні/клініки:                {len(osm_ghospitals)}')
print(f'Школи:                          {len(osm_gschools)}')

shape_route_lines_gdf = gpd.read_file(gpkg_data_path, layer='route_lines')


## 1. Аналіз маршрутів

Київ має три основні типи наземного транспорту: автобуси, тролейбуси та трамваї.

In [ ]:
display(gtfs['routes'])
type_counts = gtfs['routes']['transport_type'].value_counts()
print(type_counts)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

type_counts.plot()

# Стовпчаста діаграма
colors = ['#2196F3', '#FF9800', '#4CAF50', '#9C27B0']
type_counts.plot(
    kind='bar', ax=ax1, 
    color=colors[:len(type_counts)],
    x="Тип Транспорту",
    y="Кількість маршрутів",
    edgecolor='white'
    
)

ax1.set_title('Кількість маршрутів за типом транспорту', fontsize=13)
ax1.tick_params(axis='x', rotation=0)

for bar, count in zip(ax1.patches, type_counts):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             str(count), ha='center', va='bottom', fontweight='bold')

# Кругова діаграма
type_counts.plot(kind='pie', ax=ax2, colors=colors[:len(type_counts)],
                 autopct='%1.1f%%', startangle=90,
                 wedgeprops={'edgecolor': 'white', 'linewidth': 2})
ax2.set_title('Частка маршрутів за типом', fontsize=13)
ax2.set_ylabel('')

plt.tight_layout()
plt.savefig('../outputs/01_routes_by_type.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Графік збережено: outputs/01_routes_by_type.png')

## 2. Кількість рейсів на маршрут

Показує завантаженість кожного маршруту — скільки рейсів виконується на день.

In [ ]:
# Фільтруємо тільки будні (monday=1) — репрезентативний робочий день
calendar = pd.read_csv('../gtfs_static/calendar.txt', encoding='utf-8-sig')
weekday_sids = set(calendar[calendar['monday'] == 1]['service_id'])
trips_weekday = gtfs["trips"][gtfs["trips"]['service_id'].isin(weekday_sids)]

trips_per_route = trips_weekday.merge(
    gtfs["routes"][['route_id', 'route_short_name', 'transport_type']], on='route_id'
)
trips_count = (
    trips_per_route
    .groupby(['route_id', 'route_short_name', 'transport_type'])
    .size()
    .reset_index(name='trips_count')
)

TYPE_CONFIG = [
    ('Автобус',   '#2196F3'),
    ('Тролейбус', '#FF9800'),
    ('Трамвай',   '#4CAF50'),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 15))

for ax, (ttype, color) in zip(axes, TYPE_CONFIG):
    df = (
        trips_count[trips_count['transport_type'] == ttype]
        .sort_values('trips_count', ascending=True)
    )
    bars = ax.barh(df['route_short_name'], df['trips_count'],
                   color=color, alpha=0.85, edgecolor='white', linewidth=0.4)

    for bar, val in zip(bars, df['trips_count']):
        ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
                str(val), va='center', ha='left', fontsize=8)

    median = df['trips_count'].median()
    ax.axvline(median, color='red', linestyle='--', linewidth=1.2,
               label=f'Медіана: {median:.0f}')

    ax.set_title(f'{ttype}\n({len(df)} маршрутів)', fontsize=12, fontweight='bold')
    ax.set_xlabel('Кількість рейсів (обидва напрямки, будній день)')
    ax.legend(fontsize=9)
    ax.tick_params(axis='y', labelsize=8)
    ax.set_xlim(0, df['trips_count'].max() * 1.15)
    sns.despine(ax=ax, left=True)
    ax.xaxis.grid(True, linestyle='--', alpha=0.5)
    ax.set_axisbelow(True)

fig.suptitle('Кількість рейсів на маршрут за типом транспорту (будній день)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../outputs/02_trips_per_route.png', dpi=150, bbox_inches='tight')
plt.show()
print('Графік збережено: outputs/02_trips_per_route.png')

## 3. Інтенсивність руху по годинах доби

Визначаємо пікові години навантаження на транспортну мережу.

In [ ]:
# Годинний розподіл відправлень (тільки звичайні години 0-24)
valid_times = gtfs['stop_times'][gtfs['stop_times']['departure_sec'].notna()].copy()
valid_times = valid_times[valid_times['departure_sec'] < 86400]  # до 24:00
valid_times['hour'] = (valid_times['departure_sec'] // 3600).astype(int)

hourly_counts = valid_times.groupby('hour').size()

fig, ax = plt.subplots(figsize=(13, 5))
bars = ax.bar(hourly_counts.index, hourly_counts.values,
              color=['#FF5722' if h in range(7,10) or h in range(17,20)
                     else '#2196F3' for h in hourly_counts.index],
              edgecolor='white', linewidth=0.5)

ax.set_title('Кількість відправлень по годинах доби', fontsize=13)
ax.set_xlabel('Година доби')
ax.set_ylabel('Кількість відправлень')
ax.set_xticks(range(0, 25))

# Позначаємо пікові години
ax.axvspan(7, 9, alpha=0.15, color='red', label='Ранковий пік (7–9)')
ax.axvspan(17, 19, alpha=0.15, color='orange', label='Вечірній пік (17–19)')
ax.legend()

plt.tight_layout()
plt.savefig('../outputs/03_hourly_trips.png', dpi=150, bbox_inches='tight')
plt.show()

peak_morning = hourly_counts[7:9].mean()
peak_evening = hourly_counts[17:19].mean()
print(f'Середнє відправлень у ранковий пік:  {peak_morning:.0f}')
print(f'Середнє відправлень у вечірній пік: {peak_evening:.0f}')

## 4. Кількість зупинок на маршрут

Реальна кількість фізичних зупинок по кожному маршруту — на основі OSM-зупинок,
прив'язаних до маршрутів через просторове з'єднання з формами маршрутів (notebook 01).

> GTFS містить лише ~9 контрольних точок на маршрут — це точки де прописано час, а не всі зупинки.
> OSM-зіставлення дає ~23 реальні зупинки на маршрут.

In [ ]:
import json

# Завантажуємо OSM-зіставлені зупинки маршрутів (створено в notebook 01)
with open('../outputs/route_stops_all.json', encoding='utf-8') as f:
    route_stops_data = json.load(f)

# Для кожного маршруту беремо середню кількість зупинок між двома напрямками
route_summary = {}
for entry in route_stops_data:
    key = (entry['transport_type'], entry['route_number'])
    n = len(entry['stops'])
    if key not in route_summary:
        route_summary[key] = []
    route_summary[key].append(n)

stops_per_route = {k: sum(v) / len(v) for k, v in route_summary.items()}
stops_series = pd.Series(list(stops_per_route.values()))

print(f'Маршрутів: {len(stops_series)}')
print(f'Середня кількість зупинок на маршрут (OSM): {stops_series.mean():.1f}')
print(f'Мінімум: {stops_series.min():.0f},  Максимум: {stops_series.max():.0f}')
print()

# Порівняння GTFS vs OSM
gtfs_stops_per_trip = gtfs['stop_times'].groupby('trip_id')['stop_id'].nunique()
print(f'Для порівняння — GTFS контрольні точки:')
print(f'  Середня кількість на рейс: {gtfs_stops_per_trip.mean():.1f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Гістограма OSM-зупинок
axes[0].hist(stops_series, bins=20, color='#4CAF50', edgecolor='white')
axes[0].axvline(stops_series.mean(), color='red', linestyle='--',
                label=f'Середнє: {stops_series.mean():.1f}')
axes[0].set_title('Розподіл кількості зупинок на маршрут\n(OSM-зіставлення)', fontsize=12)
axes[0].set_xlabel('Кількість зупинок')
axes[0].set_ylabel('Кількість маршрутів')
axes[0].legend()

# Порівняння GTFS vs OSM
categories = ['GTFS\n(контрольні точки)', 'OSM\n(реальні зупинки)']
means = [gtfs_stops_per_trip.mean(), stops_series.mean()]
bars = axes[1].bar(categories, means, color=['#FF9800', '#4CAF50'], edgecolor='white', width=0.4)
for bar, val in zip(bars, means):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{val:.1f}', ha='center', fontweight='bold', fontsize=13)
axes[1].set_title('GTFS vs OSM: середня кількість зупинок', fontsize=12)
axes[1].set_ylabel('Середня кількість зупинок')
axes[1].set_ylim(0, stops_series.mean() * 1.4)

plt.tight_layout()
plt.savefig('../outputs/04_stops_per_route.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Інтерактивна карта зупинок, лікарень і шкіл

Візуалізуємо просторове покриття транспортної мережі.

In [ ]:
# Центр Києва (Майдан Незалежності)

m = folium.Map(location=KYIV_CENTER, zoom_start=11, tiles='CartoDB positron')

# Кордони Києва з OSM
kyiv_boundary = ox.geocode_to_gdf('Kyiv, Ukraine')
folium.GeoJson(
    kyiv_boundary.__geo_interface__,
    name='Кордони Києва',
    style_function=lambda f: {
        'color': '#1565C0',
        'weight': 2.5,
        'fillOpacity': 0.0,
        'dashArray': '6 4'
    }
).add_to(m)

# Шар зупинок OSM — реальні фізичні зупинки (show=True за замовчуванням)
osm_stop_group = folium.FeatureGroup(name='Зупинки OSM (всі)', show=True)
for _, row in osm['gstops'].iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=3,
        color='#FF6F00',
        fill=True,
        fill_color='#FF6F00',
        fill_opacity=0.7,
        popup=str(row.get('name', 'Зупинка'))
    ).add_to(osm_stop_group)
osm_stop_group.add_to(m)

# Шар зупинок KPT — контрольні точки розкладу (show=False, бо OSM повніший)
# stop_group = folium.FeatureGroup(name='Зупинки KPT (контрольні точки)', show=False)
# for _, row in stops_gdf.iterrows():
#     folium.CircleMarker(
#         location=[row.geometry.y, row.geometry.x],
#         radius=2,
#         color='#2196F3',
#         fill=True,
#         fill_opacity=0.6,
#         popup=str(row.get('stop_name', ''))
#     ).add_to(stop_group)
# stop_group.add_to(m)

# Шар станцій метро
metro_group = folium.FeatureGroup(name='Станції метро', show=True)
for _, row in osm['gmetro'].iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=7,
        color='#6A1B9A',
        fill=True,
        fill_color='#6A1B9A',
        fill_opacity=0.9,
        popup=str(row.get('name', 'Метро'))
    ).add_to(metro_group)
metro_group.add_to(m)

# Шар лікарень
hosp_group = folium.FeatureGroup(name='Лікарні та клініки', show=True)
for _, row in osm['ghospitals'].iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=2,
        color='#F44336',
        fill=True,
        fill_color='#F44336',
        fill_opacity=0.8,
        popup=str(row.get('name', 'Лікарня'))
    ).add_to(hosp_group)
hosp_group.add_to(m)

# Шар шкіл
school_group = folium.FeatureGroup(name='Школи', show=False)
for _, row in osm['gschools'].iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=2,
        color='#4CAF50',
        fill=True,
        fill_color='#4CAF50',
        fill_opacity=0.8,
        popup=str(row.get('name', 'Школа'))
    ).add_to(school_group)
school_group.add_to(m)

folium.LayerControl().add_to(m)
m.save('../outputs/eda_map.html')
print('Карту збережено: outputs/eda_map.html')
m


## 6. Покриття об'єктів зупинками

Перевіряємо, скільки лікарень і шкіл знаходяться в радіусі 500 м від зупинки.

In [ ]:
# Проектуємо у метричну систему (EPSG:32636 — UTM для України)
stops_proj = osm['gstops'].to_crs('EPSG:32636')
hospitals_proj = osm['ghospitals'].to_crs('EPSG:32636')
schools_proj = osm['gschools'].to_crs('EPSG:32636')

# Буфер 500 м навколо кожної зупинки
stops_buffer = stops_proj.buffer(500).unary_union

# Лікарні в зоні покриття
hosp_covered = hospitals_proj[hospitals_proj.within(stops_buffer)]
school_covered = schools_proj[schools_proj.within(stops_buffer)]

print('=== Покриття зупинками (радіус 500 м) ===')
print(f'Лікарні: {len(hosp_covered)}/{len(hospitals_proj)} ({100*len(hosp_covered)/len(hospitals_proj):.1f}%)')
print(f'Школи:   {len(school_covered)}/{len(schools_proj)} ({100*len(school_covered)/len(schools_proj):.1f}%)')

## 6. Карта маршрутів транспорту

Візуалізуємо всі маршрути за типом транспорту для перевірки коректності даних.
Кожен тип транспорту — окремий шар з вмикачем. За замовчуванням увімкнено трамваї та тролейбуси (менше ліній),
автобуси вимкнено (багато ліній — може гальмувати браузер).

In [ ]:
from folium.plugins import GroupedLayerControl
from branca.element import Element

ROUTE_TYPE_NAMES = {0: 'Трамвай', 3: 'Автобус', 11: 'Тролейбус', 1: 'Метро'}
ROUTE_TYPE_COLORS = {0: '#E53935', 3: '#1E88E5', 11: '#43A047', 1: '#8E24AA'}

m_routes = folium.Map(location=KYIV_CENTER, zoom_start=11, tiles='CartoDB positron')

m_routes.get_root().html.add_child(Element("""
<style>
.leaflet-control-layers { max-height: 85vh; overflow-y: auto; font-size: 13px; }
</style>
"""))

grouped = {}

for route_type, color in ROUTE_TYPE_COLORS.items():
    subset = shape_route_lines_gdf[shape_route_lines_gdf['route_type'] == route_type]
    if subset.empty:
        continue
    type_name = ROUTE_TYPE_NAMES[route_type]
    grouped[type_name] = []

    # Групуємо по route_short_name — один чекбокс містить обидва напрямки
    for route_name, route_group in subset.groupby('route_short_name'):
        fg = folium.FeatureGroup(
            name=f"{type_name} №{route_name}",
            show=False
        )
        for _, row in route_group.drop_duplicates('shape_id').iterrows():
            folium.GeoJson(
                row['geometry'].__geo_interface__,
                style_function=lambda f, c=color: {
                    'color': c, 'weight': 3.5, 'opacity': 0.9
                },
                tooltip=f"{type_name} №{route_name}"
            ).add_to(fg)
        fg.add_to(m_routes)
        grouped[type_name].append(fg)

GroupedLayerControl(
    groups=grouped,
    exclusive_groups=False,
    collapsed=True
).add_to(m_routes)

m_routes.save('../outputs/routes_map.html')
print('Збережено: outputs/routes_map.html')
m_routes